# Woehler curve prediction using FKM-linear guideline 

This Notebook shows an application of the FKM linear guideline for predicting the slope and fatigue limit of the Woehler curve for a selectable material.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from pylife.strength.fkm_linear.fkm_linear_factors import (
    calc_input_parameters_material,
    calc_input_parameters_stress,
    fatigue_limit_local_chap4,
)

In the following we initialize the relevant parameters for the FKM linear guideline. These are: 

1. The tensile strength "Rm" in [MPa], and the Surface roughness "Rz" in [m*1.0e-6]

2. The relative stress gradient "G0" in [1/mm]

3. The material group "MatGroupFKM" and the material temperature group "MatGroupFKM_Temp"

4. The GJL material group "GJL_Mat", if GJL is used otherwise set to None.

4. The temperature "Temprature" in [°C]

5. The temperature group of the material "MatGroupFKM_Temp"

6. The diameter "Diameter" in [mm] and the profile  "Profile" of the component must be one of {"Rod", "Tube", "Wide sheet", "Rectangle", "Square"}

7. The FKM chapter "FKM_chap". Must be one of {'chap4', 'chap5.5'}

8. The Kf method "Kf_method" for estimating of the fatigue notch factor (chap. 4.3.1.2).
   Must be one of {'Table', 'Equation'}

9. The support method "sup_method" representing the support factor calculation after Stieler or FKM2012 or von Mises. Must then be one of  {'Stieler','V90_Mises', 'A90'}

10. The "Condition variable. Heat treatment condition for fictive width b calculation.  
Must be one of {'Hardened', 'Annealed'}, optinal then set to None

11. The "Finish" variable for surface finish polished (if used, no Rz value necessary).
Must be one of {'polished'}, optional then set to None

12. The "HardProc" variable for choosing the Surface hardening method.  
Must be one of {'Inductive hardening', 'Flame hardening', 'Case hardening','Carburizing', 'Nitriding', 'Cyaniding', 'Carbonitriding','Deep rolling', 'Shot peening'}, optional then set to None
            
13. The amplitude and meanstress. However, the amplitude will be set to 1 in the following and therefrom the meanstress is inferred using the chosen stress ratio R. 

In [ ]:
# Choose the desired R ratio and compute meanstress from it (w.r.t amplitude = 1)
R = -1
mean_stress = (1+R)/(1-R)

# Define settings that are chosen globally, meaning that they do not depend on the specific node of an FE-mesh
experiment_settings =  pd.Series({"fkm_chapter": "chap4", "Profile": "Rod","Diameter": 6.0,
                                    "sup_method": "Stieler","Condition": "Hardened","MatGroupFKM": "GJL",
                                    "MatGroupFKM_Temp": "GJL",
                                }
                            )

# Define assessment parameters. This has to be done by the user according to relevant material and use case parameters
assessment_parameters = pd.DataFrame({ "Rm" : 210, "G0" : 0.5, "MatGroupFKM" : "GJL",
                                        "amplitude" : 1.0, "meanstress" : mean_stress, "Temperature" : 50,
                                        "GJL_Mat" : "GJL-250","Kf_method" : "Table", "Rz" : 200, "S_Type" : "normal",
                                         "Profile" : "Rod", "Finish" : None, "HardProc" : "None"},
                                         index=[0]
                                    )

Having the parameters initialized, we can compute the component fatigue limit for the given R ratio and therefrom predict a Woehler curve for the component.

In [ ]:
# Compute component fatigue limit according to FKM2012 Guideline:

# 1.1 Compute quantities according fkm linear guideline which are related to the material (e.g roughness factor)
assessment_parameters = calc_input_parameters_material(experiment_settings,assessment_parameters)

# 1.2 Compute quantities according fkm linear guideline which are related to the load (e.g design factor)
assessment_parameters = calc_input_parameters_stress(experiment_settings,assessment_parameters)

# 1.2 Execute FKM-Guideline and obtain fatigue strength according to chapter 4.4
res = fatigue_limit_local_chap4(assessment_parameters)

Now we can predict the Woehler curve using the compute component fatigue limit in res["SDFKM"] and the slope using the FKM-linear guidline.

In [ ]:
# Set material parameter using the fatigue limit computed above and the slope k_1 using the FKM2012 guideline
k_1 = 5
mat = pd.Series({
    'k_1': k_1,
    'ND': 1.0e6,
    'SD': res["SDFKM"].iloc[0],
    'TN': 12.,
    'TS': 1.1
})

# Visualize Woehler curves
wc1 = mat.woehler
cyc = pd.Series(np.logspace(1, 12, 200))

for pf, style in zip([0.1, 0.5, 0.9], ['--', '-', '--']):
    load = wc1.basquin_load(cyc, failure_probability=pf)
    plt.plot(cyc,load, style)

plt.xlabel("cylces"), plt.ylabel("amplitude")
plt.loglog()

Now lets do the same for a different support method using FKM2012. Therefore, we need to take the maximum loaded surface "A90" into account.

In [ ]:
# Set the values for support method and maximum loaded surface (A90)
# Define settings that are chosen globally, meaning that they do not depend on the specific node of an FE-mesh
experiment_settings["sup_method"] = "A90"
assessment_parameters["A90"] = 20

# 1.1 Compute quantities according FKM-linear guideline which are related to the material (e.g roughness factor)
assessment_parameters = calc_input_parameters_material(experiment_settings,assessment_parameters)

# # 1.2 Compute quantities according FKM-linear guideline which are related to the load (e.g design factor)
assessment_parameters = calc_input_parameters_stress(experiment_settings,assessment_parameters)

# # 1.2 Execute FKM-Guideline and obtain fatigue strength according to chapter 4.4
res = fatigue_limit_local_chap4(assessment_parameters)

# # Set material parameter using the fatigue limit computed above and the slope k_1 using the FKM2012 guideline
k_1 = 5
mat = pd.Series({
    'k_1': k_1,
    'ND': 1.0e6,
    'SD': res["SDFKM"].iloc[0],
    'TN': 12.,
    'TS': 1.1
})

# # Visualize Woehler curves
wc1 = mat.woehler
cyc = pd.Series(np.logspace(1, 12, 200))

for pf, style in zip([0.1, 0.5, 0.9], ['--', '-', '--']):
    load = wc1.basquin_load(cyc, failure_probability=pf)
    plt.plot(cyc,load, style)

plt.xlabel("cylces"), plt.ylabel("amplitude")
plt.loglog()

Moreover, we can include measured data into the Woehler prediction using the FKM linear guidline. This can be done by adapting certain parameters before computing the fatigue limit.

In [ ]:
# Choose the desired R ratio and compute meanstress from it (w.r.t amplitude = 1)
R = -1
mean_stress = (1+R)/(1-R)
experiment_settings["sup_method"] = "Stieler"

# 1.1 First compute quantities according FKM-linear guideline which are related to the material (e.g roughness factor)
assessment_parameters = calc_input_parameters_material(experiment_settings,assessment_parameters)

# Now we can adapt certain parameters like the roughness factor or the according to measurements
assessment_parameters["Kr"] = 0.85123 # instead of 0.9139530812810884
assessment_parameters["Ks"] = 0.9 # instead of 1

# After that proceed with the guideline
# 1.2 Compute quantities according FKM-linear guideline which are related to the load (e.g design factor)
assessment_parameters = calc_input_parameters_stress(experiment_settings,assessment_parameters)

# 1.2 Execute FKM-Guideline and obtain fatigue strength according to chapter 4.4
res = fatigue_limit_local_chap4(assessment_parameters)

# Set material parameter using the fatigue limit computed above and the slope k_1 using the FKM2012 guideline
k_1 = 5
mat = pd.Series({
    'k_1': k_1,
    'ND': 1.0e6,
    'SD': res["SDFKM"].iloc[0],
    'TN': 12.,
    'TS': 1.1
})

# Visualize Woehler curves
wc1 = mat.woehler
cyc = pd.Series(np.logspace(1, 12, 200))

for pf, style in zip([0.1, 0.5, 0.9], ['--', '-', '--']):
    load = wc1.basquin_load(cyc, failure_probability=pf)
    plt.plot(cyc,load, style)

plt.xlabel("cylces"), plt.ylabel("amplitude")
plt.loglog()

Finally we demonstrate the usage of the FKM-linear guideline for surface hardened material using chapter 5.5

In [ ]:
from pylife.strength.fkm_linear.fkm_linear_factors import fatigue_limit_local_chap5

# Choose the desired R ratio and compute meanstress from it (w.r.t amplitude = 1)
R = -1
mean_stress = (1+R)/(1-R)
# Define assessment parameters. This has to be done by the user accoriding to relevant material and use case parameters
# Define settings that are chosen globally, meaning that they do not depend on the specific node of an FE-mesh
experiment_settings =  pd.Series({"fkm_chapter": "chap5.5", "Profile": "Rod","Diameter": 6.0,
                                    "sup_method": "Stieler","Condition": "Hardened","MatGroupFKM": "Steel",
                                    "MatGroupFKM_Temp": None,
                                }
                            )

assessment_parameters = pd.DataFrame({ "Rm" : 2392.5, "G0" : 1.225, "A90" : 200,"A_ref" : 500,
                                    "amplitude" : 1.0, "meanstress" : mean_stress, "Temperature" : 24,
                                    "Thermal_coefficient" : 1.0, "HV" : 725.0, "HV_core" : 230.0,
                                    "GJL_Mat" : None,"Kf_method" : "Table", "Rz" : 10, "S_Type" : "shear",
                                    "Thickness" : 2.5, "Width" : None, "K_p" : 2.0, "Finish" : None,
                                    "HardProc" : None, "V90_Mises" : None},
                                    index=[0]
                                    )

Set the surface hardening method and compute component fatigue limit

In [ ]:
# Set surface hardening method
assessment_parameters["HardProc"] = "Inductive hardening"
# 1.1 Compute quantities according fkm linear guideline which are related to the material (e.g roughness factor)
assessment_parameters = calc_input_parameters_material(experiment_settings,assessment_parameters)

# 1.2 Compute quantities according fkm linear guideline which are related to the load (e.g design factor)
assessment_parameters = calc_input_parameters_stress(experiment_settings,assessment_parameters)

# 1.2 Execute FKM-Guideline and obtain fatigue strength according to chapter 4.4
res = fatigue_limit_local_chap5(assessment_parameters)

# Set material parameter using the fatigue limit computed above and the slope k_1 using the FKM2012 guideline
k_1 = 5
mat = pd.Series({
    'k_1': k_1,
    'ND': 1.0e6,
    'SD': res["SDFKM_RS"].iloc[0],
    'TN': 12.,
    'TS': 1.1
})

# Visualize Woehler curves
wc1 = mat.woehler
cyc = pd.Series(np.logspace(1, 12, 200))

for pf, style in zip([0.1, 0.5, 0.9], ['--', '-', '--']):
    load = wc1.basquin_load(cyc, failure_probability=pf)
    plt.plot(cyc,load, style)

plt.xlabel("cylces"), plt.ylabel("amplitude")
plt.loglog()


Visualize Woehler Curve